In [69]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[06/24/26 17:19:23] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=124103;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=917111;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [70]:
unis_estaca_presupuesto = catalog.load('unis_estaca_presupuesto')
unis_estaca = catalog.load('unis_estaca')['UNIS']
unis_estaca_id_inconcert = unis_estaca.loc[:,['ID_Inconcert', 
                                              'No. de documento', 
                                              'Correo 2',
                                              'Usuario Institucional',
                                              'Programa',
                                              'Etapa Studen Journey', 
                                              'Celular', 
                                              'Correo Personal', 
                                              'Column 24',
                                              'Doc ID', 
                                              'Validación']]
unis_estaca_presupuesto_completo = pd.merge(unis_estaca_presupuesto,
                                            unis_estaca_id_inconcert.drop_duplicates(subset= ['Usuario Institucional','Programa']),
                                            left_on = ['Usuario Institucional','Programa'],
                                            right_on = ['Usuario Institucional','Programa'],
                                            how = 'left',
                                            indicator= True)
unis_estaca_presupuesto_completo = unis_estaca_presupuesto_completo.loc[:,unis_estaca.columns]
unis_estaca_presupuesto_completo.to_excel('data/01_raw/unis_estaca_presupuesto_completo.xlsx', sheet_name='UNIS')

[06/24/26 17:19:24] INFO     Loading data from unis_estaca_presupuesto                         ]8;id=297494;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=448808;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (GoogleSheetsDataset)...                                                              

[06/24/26 17:19:26] INFO     Loading data from unis_estaca (ExcelDataset)...                   ]8;id=316885;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=415776;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

# Insumo

In [80]:
unis_estaca = catalog.load('unis_estaca')['UNIS']
unis_calendario_extendido_presupuesto = catalog.load('unis_calendario_extendido_presupuesto')

[06/24/26 17:21:22] INFO     Loading data from unis_estaca (ExcelDataset)...                   ]8;id=979799;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=557170;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

[06/24/26 17:21:28] INFO     Loading data from unis_calendario_extendido_presupuesto           ]8;id=972032;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=44156;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [81]:
fecha_ingreso = ['2024-08-27 00:00:00', 
                 '2025-01-14 00:00:00', 
                 '2025-05-13 00:00:00',
                 '2025-09-02 00:00:00']
cohortes = [9243, 9251, 9252, 9253]
col_fechas = ['Fecha de Baja T'
'Fecha de Baja D'
'Fecha de Reingreso'
'Fecha de Grado'
]

In [82]:
unis_estaca_presupuesto = unis_estaca[unis_estaca['Cohorte'].isin(fecha_ingreso)]

                    WARNING  C:\Users\User\AppData\Local\Temp\ipykernel_1252\2130283093.py:1:       ]8;id=805951;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=267616;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             FutureWarning: The behavior of 'isin' with dtype=datetime64 and                       
                             castable values (e.g. strings) is deprecated. In a future version,                    
                             these will not be considered matching by isin. Explicitly cast to the                 
                             appropriate dtype before calling isin instead.                                        
                               unis_estaca_presupuesto =                                                           
                             unis_estaca[unis_estaca['Cohorte'].isin(fecha_ingreso)]                               
                                                                                                                   

In [83]:
fecha_presupuesto =  datetime.strptime('2025-09-30', "%Y-%m-%d")


In [84]:
unis_estaca_presupuesto.loc[unis_estaca_presupuesto['Fecha de Baja T'] > fecha_presupuesto, 'Fecha de Baja T'] = None
unis_estaca_presupuesto.loc[unis_estaca_presupuesto['Fecha de Baja D'] > fecha_presupuesto, 'Fecha de Baja D'] = None
unis_estaca_presupuesto.loc[unis_estaca_presupuesto['Fecha de Grado'] > fecha_presupuesto, 'Fecha de Grado'] = None

In [85]:
unis_estaca_presupuesto.to_excel('data/01_raw/unis_estaca_presupuesto_completo.xlsx', sheet_name='UNIS')